# NB_03_Gold_Model
Builds the star schema the semantic model reads in Direct Lake mode, and answers the three business questions:

1. **Which suppliers can we rely on?** `fact_shipment` – delay days, fill rate, OTIF
2. **What do products really cost us once they land?** `fact_shipment` landed cost -> `fact_sales` margin
3. **Who has actually paid?** `fact_payment` (bank lines matched to invoices) -> `fact_invoice` (receivables)

Gold is small, so it is rebuilt in full on every run from the incrementally maintained silver tables.

In [ ]:
table_format = "delta"
on_time_tolerance_days = 3

In [ ]:
from pyspark.sql import functions as F, Window as W

def save(df, table):
    df.write.format(table_format).mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table:16} {spark.table(table).count():>8,} rows")

def silver(name):
    return spark.table(f"silver_{name}").drop("_batch")

## Dimensions

In [ ]:
save(silver("suppliers"), "dim_supplier")
save(silver("products"), "dim_product")
save(silver("customers"), "dim_customer")

lo, hi = spark.table("silver_fx_rates").agg(F.min("rate_date"), F.max("rate_date")).first()
dim_date = (spark.sql(f"SELECT explode(sequence(date'{lo.year}-01-01', date'{hi.year}-12-31', interval 1 day)) AS date")
    .select(
        "date",
        F.year("date").alias("year"),
        F.concat(F.lit("Q"), F.quarter("date")).alias("quarter"),
        F.month("date").alias("month_no"),
        F.date_format("date", "MMM").alias("month_name"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.weekofyear("date").alias("iso_week"),
        F.date_format("date", "EEE").alias("day_name"),
        (F.dayofweek("date").isin(1, 7)).alias("is_weekend")))
save(dim_date, "dim_date")

## fact_shipment – supplier reliability and landed cost
One row per purchase order (each PO ships as one consignment).
Landed cost = goods value converted at the PO-date FX rate + freight + import duty.

In [ ]:
fx = silver("fx_rates")
po = silver("purchase_orders")
shp = silver("shipments")

fact_shipment = (po.join(shp, "po_id", "left")
    .join(fx, (po.po_date == fx.rate_date) & (po.currency == fx.currency), "left").drop(fx.currency)
    .withColumn("fx_rate_to_gbp", F.when(F.col("currency") == "GBP", F.lit(1.0)).otherwise(F.col("rate_to_gbp")))
    .withColumn("status", F.coalesce("status", F.lit("Not Shipped")))
    # what the supplier promised: ship on the agreed date, then the planned transit time
    .withColumn("promised_arrival_date", F.date_add("expected_ship_date", F.datediff("eta_date", "ship_date")))
    .withColumn("delay_days", F.datediff("arrival_date", "promised_arrival_date"))
    .withColumn("ship_delay_days", F.datediff("ship_date", "expected_ship_date"))
    .withColumn("port_delay_days", F.datediff("arrival_date", "eta_date"))
    .withColumn("lead_time_days", F.datediff("arrival_date", "po_date"))
    .withColumn("is_on_time", F.col("delay_days") <= on_time_tolerance_days)
    .withColumn("is_in_full", F.col("qty_received") >= F.col("qty_ordered"))
    .withColumn("is_otif", F.col("is_on_time") & F.col("is_in_full"))
    .withColumn("goods_value_gbp", (F.col("qty_received") * F.col("unit_cost") * F.col("fx_rate_to_gbp")).cast("decimal(18,2)"))
    .withColumn("landed_cost_gbp", F.when(F.col("status") == "Arrived",
                F.col("goods_value_gbp") + F.col("freight_cost_gbp") + F.col("duty_gbp")))
    .withColumn("landed_unit_cost_gbp", (F.col("landed_cost_gbp") / F.col("qty_received")).cast("decimal(18,4)"))
    .select("po_id", "shipment_id", "supplier_id", "sku", "container_no", "port", "status",
            "po_date", "expected_ship_date", "ship_date", "eta_date", "promised_arrival_date", "arrival_date",
            "qty_ordered", "qty_received", "unit_cost", "currency", "fx_rate_to_gbp",
            "ship_delay_days", "port_delay_days", "delay_days", "lead_time_days",
            "is_on_time", "is_in_full", "is_otif",
            "goods_value_gbp", "freight_cost_gbp", "duty_gbp", "landed_cost_gbp", "landed_unit_cost_gbp"))
save(fact_shipment, "fact_shipment")

## fact_sales – margin after landed cost
Each SKU is costed at its weighted-average landed unit cost across everything that has arrived.
SKUs with no arrival yet fall back to supplier list cost at the latest FX rate.

In [ ]:
sku_cost = (spark.table("fact_shipment").filter("status = 'Arrived'")
    .groupBy("sku")
    .agg((F.sum("landed_cost_gbp") / F.sum("qty_received")).alias("wac_landed")))

latest_fx = (fx.withColumn("rn", F.row_number().over(W.partitionBy("currency").orderBy(F.col("rate_date").desc())))
    .filter("rn = 1").select("currency", F.col("rate_to_gbp").alias("latest_rate")))
list_cost = (silver("products").join(silver("suppliers").select("supplier_id", "currency"), "supplier_id")
    .join(latest_fx, "currency", "left")
    .select("sku", (F.col("unit_cost_ccy") * F.coalesce("latest_rate", F.lit(1.0))).alias("list_cost_gbp")))

lines = silver("invoice_lines")
fact_sales = (lines.join(sku_cost, "sku", "left").join(list_cost, "sku", "left")
    .withColumn("net_amount_gbp",
                F.round(F.col("qty") * F.col("unit_price_gbp") * (1 - F.col("discount_pct") / 100), 2).cast("decimal(18,2)"))
    .withColumn("unit_landed_cost_gbp", F.coalesce("wac_landed", "list_cost_gbp").cast("decimal(18,4)"))
    .withColumn("landed_cost_gbp", (F.col("qty") * F.col("unit_landed_cost_gbp")).cast("decimal(18,2)"))
    .withColumn("gross_margin_gbp", (F.col("net_amount_gbp") - F.col("landed_cost_gbp")).cast("decimal(18,2)"))
    .select("invoice_id", "line_no", "invoice_date", "due_date", "customer_id", "sku", "qty",
            "unit_price_gbp", "discount_pct", "net_amount_gbp", "unit_landed_cost_gbp",
            "landed_cost_gbp", "gross_margin_gbp"))
save(fact_sales, "fact_sales")

## fact_payment – bank-to-invoice reconciliation
Bank lines arrive with free-text references. Matching runs in two passes:

1. **Reference** – pull an invoice number out of the reference (`inv 123`, `INV_000123`, `Payment INV000123 thanks` ...)
2. **Name + amount** – for what is left, match payer name to customer name (normalised) where the amount equals
   exactly one still-open invoice, and that invoice has exactly one candidate payment

Anything else stays **Unmatched** for a human to review.

In [ ]:
def name_key(c):
    c = F.upper(c)
    c = F.regexp_replace(c, "&", " AND ")
    c = F.regexp_replace(c, "[^A-Z0-9 ]", "")
    c = F.regexp_replace(c, r"\bLIMITED\b", "LTD")
    return F.trim(F.regexp_replace(c, r"\s+", " "))

invoices = (spark.table("fact_sales")
    .groupBy("invoice_id", "customer_id")
    .agg(F.min("invoice_date").alias("invoice_date"), F.max("due_date").alias("due_date"),
         F.sum("net_amount_gbp").cast("decimal(18,2)").alias("invoice_amount_gbp")))

bank = (silver("bank_transactions")
    .withColumn("ref_no", F.regexp_extract(F.upper("reference"), r"INV[\s\-_#]*0*(\d{1,7})", 1))
    .withColumn("ref_invoice_id", F.when(F.col("ref_no") != "", F.concat(F.lit("INV-"), F.lpad("ref_no", 6, "0")))))

# Pass 1: reference
inv1 = invoices.select(F.col("invoice_id").alias("m_invoice_id"), F.col("invoice_date").alias("m_invoice_date"))
pass1 = (bank.join(inv1, (bank.ref_invoice_id == inv1.m_invoice_id) & (bank.txn_date >= inv1.m_invoice_date), "left")
    .withColumn("match_method", F.when(F.col("m_invoice_id").isNotNull(), F.lit("Reference"))))
matched1 = pass1.filter("match_method IS NOT NULL").select("txn_id", F.col("m_invoice_id").alias("invoice_id"), "match_method")

# Pass 2: name + amount, one-to-one only
open_inv = (invoices.join(matched1.select("invoice_id").distinct(), "invoice_id", "left_anti")
    .join(silver("customers").select("customer_id", name_key(F.col("customer_name")).alias("cust_key")), "customer_id"))
left = (pass1.filter("match_method IS NULL AND amount_gbp > 0")
    .select("txn_id", "txn_date", "amount_gbp", name_key(F.col("payer_name")).alias("payer_key")))
cand = (left.join(open_inv, (left.payer_key == open_inv.cust_key)
                  & (left.amount_gbp == open_inv.invoice_amount_gbp)
                  & (left.txn_date >= open_inv.invoice_date))
    .withColumn("n_inv", F.count("*").over(W.partitionBy("txn_id")))
    .withColumn("n_txn", F.count("*").over(W.partitionBy("invoice_id"))))
matched2 = (cand.filter("n_inv = 1 AND n_txn = 1")
    .select("txn_id", "invoice_id", F.lit("Name + Amount").alias("match_method")))

matches = matched1.unionByName(matched2)
fact_payment = (silver("bank_transactions").join(matches, "txn_id", "left")
    .join(invoices.select("invoice_id", "customer_id"), "invoice_id", "left")
    .withColumn("match_method", F.coalesce("match_method", F.lit("Unmatched")))
    .withColumn("is_matched", F.col("match_method") != "Unmatched")
    .select("txn_id", "txn_date", "amount_gbp", "payer_name", "reference",
            "invoice_id", "customer_id", "match_method", "is_matched"))
save(fact_payment, "fact_payment")

## fact_invoice – receivables position
Snapshot date = the latest date seen in the data.

In [ ]:
snapshot = spark.table("fact_payment").agg(F.max("txn_date")).first()[0]
paid = (spark.table("fact_payment").filter("is_matched")
    .groupBy("invoice_id")
    .agg(F.sum("amount_gbp").alias("paid_amount_gbp"), F.max("txn_date").alias("last_payment_date")))

fact_invoice = (invoices.join(paid, "invoice_id", "left")
    .withColumn("paid_amount_gbp", F.coalesce("paid_amount_gbp", F.lit(0)).cast("decimal(18,2)"))
    .withColumn("outstanding_gbp", F.greatest(F.col("invoice_amount_gbp") - F.col("paid_amount_gbp"), F.lit(0)).cast("decimal(18,2)"))
    .withColumn("snapshot_date", F.lit(snapshot))
    .withColumn("status",
        F.when(F.col("outstanding_gbp") <= 0.01, "Paid")
         .when(F.col("paid_amount_gbp") > 0, "Short-paid")
         .when(F.col("due_date") < F.col("snapshot_date"), "Overdue")
         .otherwise("Open"))
    .withColumn("days_to_pay", F.when(F.col("paid_amount_gbp") > 0, F.datediff("last_payment_date", "invoice_date")))
    .withColumn("days_overdue",
        F.when((F.col("outstanding_gbp") > 0.01) & (F.col("due_date") < F.col("snapshot_date")),
               F.datediff("snapshot_date", "due_date")).otherwise(F.lit(0)))
    .withColumn("aging_bucket",
        F.when(F.col("outstanding_gbp") <= 0.01, "Settled")
         .when(F.col("days_overdue") == 0, "Not due")
         .when(F.col("days_overdue") <= 30, "1-30 days")
         .when(F.col("days_overdue") <= 60, "31-60 days")
         .when(F.col("days_overdue") <= 90, "61-90 days")
         .otherwise("90+ days")))
save(fact_invoice, "fact_invoice")

## Run summary

In [ ]:
spark.table("fact_payment").groupBy("match_method").agg(
    F.count("*").alias("bank_lines"), F.sum("amount_gbp").alias("amount_gbp")).orderBy("match_method").show()
spark.table("fact_invoice").groupBy("status").agg(
    F.count("*").alias("invoices"), F.sum("outstanding_gbp").alias("outstanding_gbp")).orderBy("status").show()